# ML-04 — Search Intelligence Data Contract

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DevisriprasadSoma/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines the data contract for **Lane 2: Refresh / Content Opportunity Scoring** and verifies all claims using SQL queries over the 79M-row warehouse release via DuckDB.

---

In [ ]:
# Setup libraries
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [ ]:
import os
import getpass
import duckdb
import pandas as pd

# Authenticate with Hugging Face
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not HF_TOKEN:
    try:
        # Only prompt if in interactive session
        import sys
        if sys.stdin.isatty():
            HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
    except Exception:
        pass

con = duckdb.connect()
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    print("HF Token configured. DuckDB ready to query warehouse.")
else:
    print("No HF Token configured. Run the cell below or set up Colab secrets to load real data.")

## 1. Unit of analysis + time window

### Plain-Words Contract Answers

1. **What one row means for my lane:**
   For *Lane 2: Refresh / Content Opportunity Scoring*, one row in our final feature table represents a single **pseudonymized content item (page) for a specific client aggregated over a monthly snapshot window** (e.g. trailing-90-days relative to the partition date).

2. **Which table(s) I'll use:**
   * `dim_content` (for article properties, metadata, and keyword intent)
   * `fact_content_daily_performance` (for daily GSC impressions, clicks, avg position, and GA4 engagement)
   * `dim_clients` (to check the coverage windows per client)

3. **Which time window:**
   We use a mid-panel month partition: **March 2026** (using the partition `month=2026-03` on `fact_content_daily_performance`). This avoids using the final month (June 2026), which acts as our sealed test month.

4. **What I'd predict or rank (label or proxy):**
   We predict a binary classification label: `is_declining_label` (1 when a page's GSC impressions drop by more than 20% month-over-month, calculated as `impressions_last_30d < 0.8 * impressions_prev_30d`).

5. **One thing I deliberately exclude and why:**
   We exclude `trend_direction` and `trend_pct` from features because they contain the exact math of the label (100% data leakage). We also exclude client and page IDs (`client_hash_id`, `content_hash_id`) as features to prevent the model from memorizing specific clients rather than learning general patterns.

In [ ]:
# Code check: Verify the dataset shapes and table availability
if HF_TOKEN:
    for name, src in TABLES.items():
        # Limit row checks to quick count scans
        n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
        print(f'{name:22} {n:>12,} rows')
else:
    print("Mock/Verification Output placeholder (requires active HF_TOKEN to print real row counts).")

## 2. Fields: feature / label / context / excluded

We classify the columns we touch into four distinct buckets to prevent data leakage and guarantee that every feature is knowable at the decision moment.

*   **Feature (Safe to model):**
    *   `pos_prev30` (Historical mean GSC position over days 31–60 back)
    *   `ctr_prev30` (Clicks/impressions over days 31–60 back)
    *   `word_count` (Length of the page content from `dim_content`)
    *   `days_since_last_update` (How long since the last page refresh)
    *   `ga4_engaged_sessions` (Engagement metric from Google Analytics over days 31-60)

*   **Label (Target):**
    *   `is_declining_label` (Derived from GSC impressions: `impressions_last_30d < 0.8 * impressions_prev_30d`)

*   **Context (Identifiers & grouping):**
    *   `content_hash_id` (Unique page ID, used for joins and deduplication)
    *   `client_hash_id` (Unique client ID, used for client-holdout validation splits)
    *   `report_date` (To verify snapshot windows)

*   **Excluded (Removed with reasoning):**
    *   `impressions_last_30d` & `clicks_last_30d` (Contain the outcome/label window; using them creates leakage)
    *   `provider_used` & `model_used` (LLM generation tags; excluded to prevent the model from learning brand-specific biases)

In [ ]:
# Code check: Verify dim_content metadata columns
if HF_TOKEN:
    df_cols = con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 1").df()
    print("Available metadata columns in dim_content:")
    print(df_cols.columns.tolist())
else:
    print("Mock/Verification Output placeholder (requires HF_TOKEN).")

## 3. Verify it with queries (grain, counts, missing values, windows)

We write three specific queries on the mid-panel month partition (`month=2026-03`) to verify the data contract claims.

In [ ]:
if HF_TOKEN:
    # Query 1: Verify the daily grain of the fact table
    # Grouping by report_date, client, and content should return c = 1 for all rows.
    grain_check = con.sql(f"""
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2, 3
        HAVING c > 1
        LIMIT 5
    """).df()
    print(f"Grain check returned {len(grain_check)} rows (expect 0 if grain holds):")
    print(grain_check)
else:
    print("Query 1: Grain verification (requires HF_TOKEN).")

In [ ]:
if HF_TOKEN:
    # Query 2: Verify active page counts and date spans
    counts_check = con.sql(f"""
        SELECT COUNT(DISTINCT content_hash_id) as unique_pages,
               COUNT(*) as total_daily_rows,
               MIN(report_date) as start_date,
               MAX(report_date) as end_date
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
    """).df()
    print("Row count and date span for partition month 2026-03:")
    print(counts_check)
else:
    print("Query 2: Counts & date span verification (requires HF_TOKEN).")

In [ ]:
if HF_TOKEN:
    # Query 3: Verify availability flags filtering
    # Null check for GSC and GA4 availability, and count of records with GA4 data active.
    availability_check = con.sql(f"""
        SELECT 
            COUNT(*) as total_rows,
            SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_active_rows,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_active_rows,
            SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) as both_active_rows
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
    """).df()
    print("Availability flags in partition month 2026-03:")
    print(availability_check)
else:
    print("Query 3: Availability verification (requires HF_TOKEN).")

## 4. Feature frame construction + Leakage Trap

We aggregate the warehouse table inside DuckDB to construct a feature frame for March 2026. Then, we demonstrate the feature leakage trap (adding a label-derived column on purpose to watch the score inflate, then removing it).

In [ ]:
if HF_TOKEN:
    # 1. Aggregate daily rows to monthly page-level features for March 2026
    # Target month: month='2026-03'
    # We define features on the first 15 days (prev30 window proxy) to predict the outcome on the last 15 days.
    raw_data = con.sql(f"""
        WITH monthly_agg AS (
            SELECT 
                client_hash_id,
                content_hash_id,
                -- Feature window: first 15 days of the month
                SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_impressions ELSE 0 END) as imp_prev15,
                SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_clicks ELSE 0 END) as clk_prev15,
                AVG(CASE WHEN DAY(report_date) <= 15 THEN gsc_avg_position END) as pos_prev15,
                SUM(CASE WHEN DAY(report_date) <= 15 THEN ga4_pageviews ELSE 0 END) as views_prev15,
                
                -- Outcome window: last 15 days of the month
                SUM(CASE WHEN DAY(report_date) > 15 THEN gsc_impressions ELSE 0 END) as imp_last15
            FROM {TABLES['fact_daily']}
            WHERE month = '2026-03'
              AND gsc_data_available IS TRUE
            GROUP BY 1, 2
            HAVING imp_prev15 >= 50 -- volume floor filter
        )
        SELECT 
            m.*,
            c.content_type,
            c.word_count
        FROM monthly_agg m
        LEFT JOIN {TABLES['dim_content']} c ON m.content_hash_id = c.content_hash_id
    """).df()
    
    # Calculate historical CTR feature
    raw_data['ctr_prev15'] = (raw_data['clk_prev15'] / raw_data['imp_prev15'] * 100).fillna(0)
    # Define target label: decline of more than 20% in impressions
    raw_data['is_declining_label'] = (raw_data['imp_last15'] < 0.8 * raw_data['imp_prev15']).astype(int)
    
    print(f"Feature frame constructed: {len(raw_data):,} content items.")
    print(raw_data.head())
else:
    print("Feature frame construction placeholder (requires HF_TOKEN).")

### The Trap: Demonstrating Feature Leakage

We inject `imp_last15` (the sum of impressions in the outcome window) into the features list. Since the label `is_declining_label` is mathematically derived from `imp_last15`, this is a classic leakage trap that will artificially inflate model metrics to ~100% accuracy.

In [ ]:
if HF_TOKEN:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    # Prepare dataset, filling nulls in metadata
    df_model = raw_data.copy()
    df_model['word_count'] = df_model['word_count'].fillna(df_model['word_count'].median())
    
    honest_features = ['imp_prev15', 'pos_prev15', 'views_prev15', 'ctr_prev15', 'word_count']
    leaked_features = honest_features + ['imp_last15'] # Injected leakage feature
    
    y = df_model['is_declining_label']
    
    # Split
    X_train_h, X_test_h, y_train, y_test = train_test_split(df_model[honest_features], y, test_size=0.3, random_state=42)
    X_train_l, X_test_l, _, _ = train_test_split(df_model[leaked_features], y, test_size=0.3, random_state=42)
    
    # Train leaked model
    model_leaked = RandomForestClassifier(random_state=42).fit(X_train_l, y_train)
    pred_leaked = model_leaked.predict(X_test_l)
    score_leaked = accuracy_score(y_test, pred_leaked)
    
    # Train honest model
    model_honest = RandomForestClassifier(random_state=42).fit(X_train_h, y_train)
    pred_honest = model_honest.predict(X_test_h)
    score_honest = accuracy_score(y_test, pred_honest)
    
    print("=== LEAKAGE EXPERIMENT RESULTS ===")
    print(f"Leaked model accuracy (including imp_last15): {score_leaked:.4f} (Too good to be true)")
    print(f"Honest model accuracy (only features knowable before day 15): {score_honest:.4f}")
    print("==================================")
    
    # Crucial: Delete the leaked model and features to keep the notebook clean and honest!
    del model_leaked, X_train_l, X_test_l
    print("Leaked model and features removed from runtime memory.")
else:
    print("Leakage experiment placeholder (requires HF_TOKEN).")

## 5. Data limits

### One Named Limitation of the Slice

**Unbalanced Panel coverage & Null availability flags:**
The search console and analytics history depth differs widely per client (some have 17 months of data, others have less than 3). Furthermore, `ga4_data_available` is not always a clean binary flag; for a significant number of records and clients, the availability flag is `NULL`. A simple binary filter like `= FALSE` or `NOT TRUE` fails to capture these records properly, introducing systematic missingness bias. All filtering must use explicit `IS TRUE` checks to be safe.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.